# 02 · Fine-tune E3 on a Colab GPU (optional)

E3 (DistilBERT) also trains on a laptop CPU: the published results in `RESULTS.md` came from `uv run txncat train finetune` run locally. Use this notebook to re-run it on a GPU.

**Before you start (on your machine, where PostgreSQL has the data):**

```bash
uv run txncat data export-encoder   # writes data/generated/encoder_input.csv
```

Then in Colab: *Runtime → Change runtime type → GPU*, and run the cells in order. This notebook was not executed in the repository; it has no stored outputs.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo 'No GPU: select a GPU runtime first'
!git clone --depth 1 https://github.com/Jedges14/txn-categorizer.git
%cd txn-categorizer
# Colab already has a CUDA build of torch; install the project and the pinned transformers only.
!pip install -q -e . 'transformers==5.24.1'

Upload `encoder_input.csv` (from `txncat data export-encoder`).

In [ ]:
import pathlib
import shutil

from google.colab import files

uploaded = files.upload()
pathlib.Path("data/generated").mkdir(parents=True, exist_ok=True)
shutil.move(next(iter(uploaded)), "data/generated/encoder_input.csv")

Rebuild the synthetic CSV (deterministic, seed 42). It's only used to flag external rows whose merchant appears in training, for the unseen-merchant subset score.

In [ ]:
!txncat data build

In [ ]:
!txncat train finetune --input data/generated/encoder_input.csv --seeds 0 1 2 --epochs 3

Download the predictions and metrics, then copy them into your local repo's `predictions/` and `reports/metrics/`.

In [ ]:
!zip -q e3_outputs.zip predictions/E3_finetune_seed*.csv reports/metrics/e3_finetune.json
files.download("e3_outputs.zip")